# EnMAP segmentation sharding — Colab runbook

Produces `enmap_seg` shards: EnMAP patches carrying the provider quality layers as
training labels for the cloud/water segmentation head.

Run the cells in order. The three stages are deliberately separated:

1. **Screen** — read every scene's cover percentages. Seconds, no cubes touched.
2. **Smoke** — shard one scene with a large stride. ~20 s. Confirms the whole path works.
3. **Run** — shard a subset for real.

Do not skip stage 2. The last two bugs in this pipeline were both invisible
until a real scene went through it.

**Before you start**

- Scenes must be on Drive as `ENMAP01-*` **folders** (not zips).
- Colab sessions die at 12–24 h. Sharding is resumable: one shard per scene,
  already-published scenes are skipped. Re-running this notebook continues.
- Shards are written to Colab's **local disk** and copied to Drive on completion.
  One large write per scene beats thousands of small writes to a network mount.

Design notes: `docs/lld/segmentation-sharding.md`.

## Setup

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# Repo + dependencies.
#
# Private repo: create a fine-grained PAT with read access and use
#   !git clone https://<TOKEN>@github.com/rashwin88/hsi-anomaly-foundations-allotrope.git
# Do not paste a token into a cell you intend to commit.

import os

REPO = '/content/hsi-anomaly-foundations-allotrope'

if not os.path.isdir(REPO):
    !git clone https://github.com/rashwin88/hsi-anomaly-foundations-allotrope.git {REPO}
else:
    !cd {REPO} && git pull --ff-only

# Only what sharding needs — torch is not required, this is CPU work.
!pip -q install rasterio pystac webdataset numexpr

In [ ]:
import sys

if REPO not in sys.path:
    sys.path.insert(0, REPO)

import numpy as np
import rasterio

from app.utils.files.enmap_scene_cover import read_scene_cover
from app.utils.patch_generation.scene_storage import LocalSceneStorage
from app.utils.patch_generation.intermediate.enmap_segmentation_patcher import (
    EnmapSegmentationSharder,
    scene_stratum,
)

print('rasterio', rasterio.__version__, '| GDAL', rasterio.__gdal_version__)
print('numpy   ', np.__version__)
print('imports OK')

## Configure

Everything you are likely to change lives in this one cell.

In [ ]:
# --- where the data is -------------------------------------------------
SCENE_ROOT = '/content/drive/MyDrive/PS_11/EnMAP_datasets/vinoth'
SHARD_DIR  = '/content/drive/MyDrive/PS_11/EnMAP_datasets/seg_shards'
WORK_DIR   = '/content/work'          # Colab local disk, NOT Drive

# --- patch geometry ----------------------------------------------------
WIDTH = HEIGHT = 128
STRIDE = 64                            # 50% overlap; raise to shrink output

# --- how much to do ----------------------------------------------------
MAX_SCENES = 40                        # cap BEFORE splitting; None = all
TEST_FRACTION = 0.2
SEED = 42

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(SHARD_DIR, exist_ok=True)

storage = LocalSceneStorage(scene_root=SCENE_ROOT, shard_dir=SHARD_DIR)
found = storage.list_scenes()
print(f'{len(found)} scene folders under {SCENE_ROOT}')
for s in found[:3]:
    print('  ', s)
if not found:
    raise SystemExit('No ENMAP01-* folders found. Check SCENE_ROOT, and that '
                     'scenes are extracted folders rather than .zip files.')

## Stage 1 — Screen the scenes

Reads only the first 64 KB of each `METADATA.XML`, where all six cover tags live.
About 3 ms per scene.

The split is **stratified on cloud**, because cloud is scarce: in a 212-scene screen
only 37 scenes contained any, none above 14%. A random split can leave a test set
with no cloud at all — which is how an earlier experiment reported a score that
rested on three scenes out of twelve.

In [ ]:
import collections

TAGS = ['cloudCover', 'cloudShadow', 'hazeCover', 'cirrusCover', 'snowCover', 'waterCover']
rows, strata = [], collections.Counter()

for sid in found:
    try:
        cover = read_scene_cover(storage.fetch_scene(sid, ''))
    except FileNotFoundError:
        print('skip (no METADATA.XML):', sid)
        continue
    rows.append(cover)
    strata[scene_stratum(cover)] += 1

print(f'screened {len(rows)} scenes\n')
print(f"{'tag':<14}{'=0':>6}{'>0':>6}{'>=5%':>7}{'>=10%':>7}{'max':>7}")
for t in TAGS:
    v = np.array([r.get(t, 0.0) for r in rows])
    print(f'{t:<14}{(v == 0).sum():>6}{(v > 0).sum():>6}{(v >= 5).sum():>7}'
          f'{(v >= 10).sum():>7}{v.max():>7.0f}')

print('\nstrata for the split:')
for k in ['cloud_high', 'cloud_low', 'snow', 'clear']:
    print(f'  {k:<12}{strata[k]:>5}   -> ~{round(strata[k] * TEST_FRACTION)} into test')

## Stage 2 — Smoke test

One scene, large stride, so it finishes in seconds and produces a handful of patches.

**What to check in the output below:**

- `pixels.npy` is `(165, 128, 128)` — **165**, not 188. If it says 188 the common
  wavelength grid is not being applied and every shard would be unusable.
- dtype is `float16`.
- all six `label_*.npy` keys are present.
- the source scene folder still exists afterwards.

In [ ]:
import shutil

SMOKE_OUT = '/content/smoke_out'
shutil.rmtree(SMOKE_OUT, ignore_errors=True)
os.makedirs(SMOKE_OUT)

smoke = EnmapSegmentationSharder(
    storage=LocalSceneStorage(scene_root=SCENE_ROOT, shard_dir=SMOKE_OUT),
    work_dir='/content/smoke_work',
    split='train',
    width=128, height=128, stride=512,   # big stride = few patches
)
print('quality masks applied:', smoke.band_filter_config.quality_masks_to_apply, '(must be [])')
print('prefix:', smoke.destination_prefix)
smoke.sharder(scenes=1)

In [ ]:
import io
import json
import tarfile

tars = sorted(os.listdir(SMOKE_OUT))
assert tars, 'no shard written — read the error above'
path = os.path.join(SMOKE_OUT, tars[0])
print(f'{tars[0]}  ({os.path.getsize(path) / 1e6:.1f} MB)\n')

with tarfile.open(path) as tf:
    names = tf.getnames()
    first = names[0].split('.', 1)[0]
    for n in [x for x in names if x.startswith(first + '.')]:
        key = n.split('.', 1)[1]
        raw = tf.extractfile(n).read()
        if key.endswith('.npy'):
            a = np.load(io.BytesIO(raw), allow_pickle=False)
            extra = ''
            if key.startswith('label_'):
                v, c = np.unique(a, return_counts=True)
                extra = '  ' + str({int(x): round(float(y) / a.size * 100, 1)
                                    for x, y in zip(v, c)})
            print(f'  {key:<24}{str(a.shape):<18}{str(a.dtype):<10}{extra}')
        else:
            m = json.loads(raw)
            print(f"  {key:<24}sensor={m['sensor']} bands={m['band_count']}")

band_count = np.load(io.BytesIO(tarfile.open(path).extractfile(
    first + '.pixels.npy').read())).shape[0]
print('\n165 bands:', 'YES' if band_count == 165 else f'NO — got {band_count}, STOP')
print('source scene intact:', os.path.isdir(storage.fetch_scene(found[0], '')))

## Stage 3 — The real run

Check the storage estimate before starting. At stride 64 a patch is ~8.2 MB
(5.4 float16 pixels + 2.7 per-band validity + 0.1 labels), and a scene yields
roughly 320 patches — about **2.6 GB per scene**.

Raise `STRIDE` if that is too much: stride 96 roughly halves it, stride 128
quarters it.

In [ ]:
n = MAX_SCENES if MAX_SCENES else len(found)
patches_per_scene = (1175 // STRIDE) * (1330 // STRIDE)      # approximate
gb = n * patches_per_scene * 8.2 / 1000

print(f'{n} scenes x ~{patches_per_scene} patches x 8.2 MB  =  ~{gb:.0f} GB')
print(f'destination: {SHARD_DIR}')

free_gb = shutil.disk_usage('/content').free / 1e9
print(f'\nColab local free: {free_gb:.0f} GB '
      f'(needs only ~5 GB — one shard at a time, deleted after publish)')
print('Drive free space is the constraint. Check it before continuing.')

In [ ]:
# Safe to re-run: scenes whose shard already exists are skipped.
from scripts.generate_segmentation_patches import run

run(
    storage=storage,
    work_dir=WORK_DIR,
    width=WIDTH,
    height=HEIGHT,
    stride=STRIDE,
    test_fraction=TEST_FRACTION,
    seed=SEED,
    max_scenes=MAX_SCENES,
)

## Verify what landed

In [ ]:
shards = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar'))
total = sum(os.path.getsize(os.path.join(SHARD_DIR, f)) for f in shards)
print(f'{len(shards)} shards, {total / 1e9:.1f} GB total\n')

label_px = collections.Counter()
n_patches = 0
for f in shards[:5]:                       # sample, not the whole set
    with tarfile.open(os.path.join(SHARD_DIR, f)) as tf:
        for n_ in tf.getnames():
            key = n_.split('.', 1)[1]
            if key == 'label_cloud.npy':
                n_patches += 1
            if key.startswith('label_') and key != 'label_classes.npy':
                a = np.load(io.BytesIO(tf.extractfile(n_).read()))
                label_px[key] += int((a > 0).sum())

px = max(n_patches, 1) * 128 * 128
print(f'across {n_patches} patches from {min(5, len(shards))} shards:')
for k, v in sorted(label_px.items()):
    print(f'  {k:<24}{v / px * 100:6.3f}% of pixels')
print('\nIf cloud is 0.000% everywhere, the sampled shards contain no cloudy '
      'scenes — check the strata table from Stage 1.')

## Stage 4 — Shuffle into training shards

The intermediate stage writes **one shard per scene** so a killed session
resumes rather than restarts. That leaves each shard holding ~270 spatially
sequential tiles of a single scene — and the trainers shuffle only at *shard*
level, so a batch would come from very few scenes.

This stage interleaves several shards at once and applies a shuffle buffer,
then writes evenly sized ~1 GB shards.

- `group_size` is **the mixing knob**, not a speed setting. It is how many
  scenes are read round-robin at a time.
- `shuffle_size` costs `shuffle_size x 8.2 MB` of RAM — 200 is ~1.6 GB.
- Inputs are **left untouched**, so verify the output before deleting them.

Needs roughly as much Drive space again as the intermediate shards.

In [ ]:
from app.utils.patch_generation.final.local_final_shuffler import LocalFinalShuffler

FINAL_DIR = SHARD_DIR.rstrip('/') + '_final'

shuffler = LocalFinalShuffler(
    source_dir=SHARD_DIR,
    dest_dir=FINAL_DIR,
    shuffle_size=200,          # x 8.2 MB ~= 1.6 GB RAM
    group_size=8,              # scenes interleaved at once — the mixing knob
    shard_size_bytes=1 << 30,  # ~1 GB per output shard
    seed=SEED,
)
print(shuffler)
shuffler.write_shards()

### Did it actually mix?

Reads the first output shard and reports how often consecutive patches come
from *different* scenes. Near 0% means something is wrong — check `group_size`.

In [ ]:
finals = sorted(f for f in os.listdir(FINAL_DIR) if f.endswith('.tar'))
total = sum(os.path.getsize(os.path.join(FINAL_DIR, f)) for f in finals)
print(f'{len(finals)} final shards, {total / 1e9:.1f} GB')

scenes_seq = []
with tarfile.open(os.path.join(FINAL_DIR, finals[0])) as tf:
    for n in tf.getnames():
        key, _, ext = n.partition('.')
        if ext == 'pixels.npy':
            scenes_seq.append(key.split('#')[0])

changes = sum(1 for a, b in zip(scenes_seq, scenes_seq[1:]) if a != b)
print(f'\nfirst shard: {len(scenes_seq)} patches from '
      f'{len(set(scenes_seq))} distinct scenes')
print(f'consecutive patches from different scenes: '
      f'{changes}/{len(scenes_seq) - 1} ({changes / max(len(scenes_seq) - 1, 1) * 100:.0f}%)')

in_count = len([f for f in os.listdir(SHARD_DIR) if f.endswith('.tar')])
print(f'\nintermediate shards still present: {in_count} '
      f'(delete only after you are happy with the above)')

## Notes

**Resuming.** Re-run the Stage 3 cell. Scenes with a shard already in `SHARD_DIR`
are skipped by name, so a killed session costs the scene in flight.

**`MAX_SCENES` vs a smaller dataset.** `MAX_SCENES` caps *before* the split, so the
strata are preserved. There is also a per-split cap (`max_scenes_per_split`) but it
truncates *after* splitting and can leave the sides lopsided — it is a smoke-test
flag, not a way to build a smaller dataset.

**Labels are stored raw.** Cirrus is 0–3 by thickness, not binary. In
`label_classes.npy` both 0 and 3 are no-data (0 = error, 3 = off-swath), 1 = land,
2 = water. Off-swath alone is ~25% of every raster and outnumbers cloud pixels
roughly 35:1, so a trainer must exclude it.

**float16.** Pixels are float16 to halve storage. **A trainer must cast to float32** —
the models are fp32.

**Cirrus will be the weak class and that is not fixable here.** The 1380 nm band that
makes cirrus obvious is absent from every Level-2 reflectance product — EnMAP blanks
it, PRISMA ships it empty, AVIRIS flags it bad. The provider's cirrus layer is derived
from radiance we never see.